# Model Transfer


<b> Objective: </b>

Test if the model transfer knowledge between regions. Testing region is 20% of Pierpaolo's data. Training regions differ between setups:

1. Pierpaolo's 80%
2. Pierpaolo's 80% + Whole Rock 100%
3. Whole Rock 100%

(1) is the baseline; (2) tests if extra knowledge from a separate region helps; (3) tests if only that extra knowledge is up-to-par.

----------------------

<i><b>Author: </b> Salvador Palma <br>
<b>Institution: </b> Copenhagen University </i>

In [2]:
import sys
import os
COLAB_SESSION = True

DRIVE_PATH = ""
if COLAB_SESSION:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_PATH = "/content/drive/MyDrive/UCPH/POOCS/"
    sys.path.insert(0, DRIVE_PATH)
    
import pandas as pd
from GreenlandUtils import *

Mounted at /content/drive


In [6]:
ELEMENTS = ["Cu", "Pb", "Zn", "Mo", "W", "Sb", "Ba", "Sr"]
RATIOS = [("Cu", "Pb"), ("Cu", "Zn"), ("Mo", "Pb"), ("Mo", "Sb"), ("W", "Pb"), ("W", "Sb"), ("Ba", "Sr")]

DATA_PATH = "Data/Pierpaolo/"
SOURCE_XLSX = "Data/Zonation/greenland_portal.xlsx"

CACHE_DATA = f"{DRIVE_PATH}{DATA_PATH}Data.csv"
TARGETS = [f"{e} ppm" for e in ELEMENTS] + [f"{a}/{b}" for a, b in RATIOS]


def Treat(raw: pd.DataFrame) -> pd.DataFrame:
    elementCols = [c for c in raw.columns if str(c).endswith(("ppm", "ppb", "wt%"))]
    df = raw.copy()

    for col in elementCols:
        values = df[col].astype(float)
        zero = values <= 0
      
        values[zero] = np.nan
        df[col] = values

    df = df.groupby("Sampleno", as_index=False).agg(
        { **{ c: "first" for c in ["Longitude", "Latitude", "Batch"] }, **{ c: "median" for c in elementCols } })
    df[elementCols] = np.log10(df[elementCols].where(df[elementCols] > 0))

    for a, b in RATIOS:
        df[f"{a}/{b}"] = df[f"{a} ppm"] - df[f"{b} ppm"]
    return df


if os.path.exists(CACHE_DATA):
    df = pd.read_csv(CACHE_DATA, delimiter=";")
    print("Loaded cached data")
else:
    raw = pd.read_excel(f"{DRIVE_PATH}{SOURCE_XLSX}")
    df = Treat(raw)
    df.to_csv(CACHE_DATA, sep=";", index=False)
    print(f"Treated {len(raw)} rows")

x, y = lonlatToUTM24N(df["Longitude"].to_numpy(float), df["Latitude"].to_numpy(float))
df["x_km"], df["y_km"] = x / 1000, y / 1000

printable = Printable()
printable.title(f"{len(df)} samples at {df[['x_km', 'y_km']].drop_duplicates().shape[0]} locations")
for target in TARGETS:
    printable.add(target, f"{df[target].notna().sum():>5} values")
printable.flush()

Loaded cached data
===== 4000 samples at 1668 locations =====
Cu ppm:       3586 values
Pb ppm:       3140 values
Zn ppm:       2462 values
Mo ppm:       1669 values
W ppm:         686 values
Sb ppm:        538 values
Ba ppm:       2171 values
Sr ppm:        862 values
Cu/Pb:        3010 values
Cu/Zn:        2356 values
Mo/Pb:        1596 values
Mo/Sb:         487 values
W/Pb:          599 values
W/Sb:          409 values
Ba/Sr:         491 values
